# 📗 02-图像处理与数据加载

> 本笔记是 PyTorch 学习路线的第二部分，涵盖图像变换、自定义数据集和数据批量加载。
>
> **学习路径**: ① [PyTorch 基础](./01-PyTorch基础.ipynb) → ② 本笔记（数据处理） → ③ [神经网络与完整训练](./03-神经网络与完整训练.ipynb)

## 📋 目录
- [1. 图像变换 torchvision.transforms](#1-图像变换-torchvisiontransforms)
- [2. 自定义数据集 Dataset](#2-自定义数据集-dataset)
- [3. CIFAR10 数据集](#3-cifar10-数据集)
- [4. DataLoader 批量加载](#4-dataloader-批量加载)
- [5. 总结与速查](#5-总结与速查)

**运行环境**：`pip install torch torchvision numpy Pillow tensorboard`

> ⚠️ `transforms` 是图像变换模块，与 Transformer 神经网络架构**完全无关**。

---

## 1. 图像变换 torchvision.transforms

### 1.1 ToTensor — 为什么需要 Tensor？

`transforms.ToTensor()` 是图像预处理中最基础的操作，它会自动完成三件事：

1. **PIL 图像 → PyTorch Tensor**（格式转换）
2. **像素值 `[0, 255]` → `[0.0, 1.0]`**（归一化）
3. **维度 `(H, W, C)` → `(C, H, W)`**（调整顺序，PyTorch 标准输入格式）

In [1]:
# ==================== ToTensor 基本用法 ====================
from torchvision import transforms
from PIL import Image
import torch

# 读取 PIL 图像
# image = Image.open("image.jpg")  # 格式: H×W×C, 值域 [0,255]

# 创建转换器
to_tensor = transforms.ToTensor()

# 执行转换
# tensor_image = to_tensor(image)  # 格式: C×H×W, 值域 [0.0, 1.0]

print("ToTensor 转换三件事:")
print("  ① PIL.Image → torch.Tensor")
print("  ② 像素值 [0,255] → [0.0, 1.0]")
print("  ③ 维度 (H,W,C) → (C,H,W)")
print("\n这是送入神经网络前的标准预处理步骤")

ToTensor 转换三件事:
  ① PIL.Image → torch.Tensor
  ② 像素值 [0,255] → [0.0, 1.0]
  ③ 维度 (H,W,C) → (C,H,W)

这是送入神经网络前的标准预处理步骤


### 1.2 常用 Transform 操作速查表

| 变换方法 | 功能说明 | 常用场景 |
|---------|---------|----------|
| `ToTensor()` | PIL→Tensor, 归一化 [0,1] | **所有图像处理必备** |
| `Normalize(mean, std)` | 按通道归一化: `(x - mean) / std` | 匹配预训练模型 |
| `Resize(size)` | 调整图像尺寸 | 统一输入尺寸 |
| `Compose(transforms)` | 组合多个变换 | 预处理流水线 |
| `RandomCrop(size)` | 随机裁剪 | 数据增强 |
| `RandomHorizontalFlip(p)` | 随机水平翻转 (概率 p) | 数据增强 |
| `ColorJitter()` | 随机调整亮度/对比度/饱和度 | 数据增强 |
| `CenterCrop(size)` | 中心裁剪 | 测试时预处理 |

In [ ]:
# ==================== Transform 操作示例 ====================
from torchvision import transforms
from PIL import Image

# 读取图像
# image = Image.open("image.jpg")

# ── 1. Normalize 归一化 ──
# 公式: output[channel] = (input[channel] - mean[channel]) / std[channel]
normalize = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
# 效果: 将 [0, 1] 区间映射到 [-1, 1] 区间
print("Normalize 示例:")
print("  公式: output = (input - mean) / std")
print("  常用: Normalize([0.5,0.5,0.5], [0.5,0.5,0.5]) 映射到 [-1,1]")

# ── 2. Resize 调整大小 ──
resize_fixed = transforms.Resize((512, 512))   # 固定尺寸
resize_short = transforms.Resize(512)           # 短边缩放到 512, 保持宽高比

# ── 3. Compose 组合变换 ──
transform_pipeline = transforms.Compose([
    transforms.Resize(256),       # 第 1 步: resize
    transforms.CenterCrop(224),   # 第 2 步: 中心裁剪
    transforms.ToTensor(),        # 第 3 步: 转 Tensor
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # 第 4 步: 归一化
])
# result = transform_pipeline(image)  # 按顺序依次执行

# ── 4. RandomCrop 随机裁剪 (数据增强) ──
random_crop = transforms.Compose([
    transforms.RandomCrop(200),
    transforms.ToTensor()
])
# 每次裁剪位置不同, 产生不同的训练样本

print("\nCompose 流水线示例:")
print("  Resize(256) → CenterCrop(224) → ToTensor() → Normalize()")

---

## 2. 自定义数据集 Dataset

继承 `torch.utils.data.Dataset` 来创建自定义数据集，必须实现两个方法：
- `__len__()` — 返回数据集大小
- `__getitem__(idx)` — 根据索引返回单个样本 (图像, 标签)

In [2]:
# ==================== 自定义数据集 ====================
from torch.utils.data import Dataset
from PIL import Image
import os

class MyImageDataset(Dataset):
    """
    自定义图像数据集
    
    目录结构:
        root_dir/
        ├── class_A/
        │   ├── img1.jpg
        │   └── img2.jpg
        └── class_B/
            ├── img3.jpg
            └── img4.jpg
    
    Args:
        root_dir: 数据集根目录
        label_dir: 类别目录名
        transform: 可选的图像变换
    """
    def __init__(self, root_dir, label_dir, transform=None):
        self.root_dir = root_dir
        self.label_dir = label_dir
        self.path = os.path.join(root_dir, label_dir)
        self.img_names = os.listdir(self.path)
        self.transform = transform
    
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.root_dir, self.label_dir, img_name)
        image = Image.open(img_path)
        label = self.label_dir
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def __len__(self):
        return len(self.img_names)

# 使用示例（需要实际数据目录）:
# dataset = MyImageDataset("data/train", "cats", transform=transforms.ToTensor())
# img, label = dataset[0]  # 获取第 0 个样本
# print(f"样本数: {len(dataset)}")

print("Dataset 三要素:")
print("  __init__:   初始化（加载文件列表、设置变换）")
print("  __len__:    返回数据集大小")
print("  __getitem__: 根据索引返回 (image, label)")

Dataset 三要素:
  __init__:   初始化（加载文件列表、设置变换）
  __len__:    返回数据集大小
  __getitem__: 根据索引返回 (image, label)


---

## 3. CIFAR10 数据集

CIFAR-10 是常用的图像分类数据集：
- **60000 张** 32×32 彩色图像
- **10 个类别**，每类 6000 张
- 50000 张训练 + 10000 张测试

In [3]:
# ==================== CIFAR10 下载与加载 ====================
import torchvision
import torchvision.transforms as transforms

# 数据预处理
transform = transforms.Compose([transforms.ToTensor()])

# 下载训练集
print("下载 CIFAR10 训练集...")
train_data = torchvision.datasets.CIFAR10(
    root="./dataSet",
    train=True,           # 训练集
    transform=transform,
    download=True         # 不存在则自动下载 (~170MB)
)
print(f"训练集: {len(train_data)} 张")

# 下载测试集
print("下载 CIFAR10 测试集...")
test_data = torchvision.datasets.CIFAR10(
    root="./dataSet",
    train=False,          # 测试集
    transform=transform,
    download=True
)
print(f"测试集: {len(test_data)} 张")

# 类别标签映射
CIFAR10_CLASSES = [
    "飞机", "汽车", "鸟类", "猫", "鹿", "狗", "青蛙", "马", "船只", "卡车"
]
print(f"\n类别: {CIFAR10_CLASSES}")

下载 CIFAR10 训练集...
训练集: 50000 张
下载 CIFAR10 测试集...
测试集: 10000 张

类别: ['飞机', '汽车', '鸟类', '猫', '鹿', '狗', '青蛙', '马', '船只', '卡车']


---

## 4. DataLoader 批量加载

DataLoader 是训练循环中的核心组件，提供批量加载、随机打乱、多进程加速等功能。

### 4.1 DataLoader 参数详解

| 参数 | 说明 | 推荐值 |
|------|------|--------|
| `dataset` | 数据源（Dataset 对象） | 必填 |
| `batch_size` | 每批样本数 | 训练: 32-128; 测试: 64-256 |
| `shuffle` | 是否打乱数据 | 训练: `True`; 测试: `False` |
| `num_workers` | 多进程数 | Linux: 2-4; **Windows: 0** |
| `drop_last` | 丢弃最后不完整批次 | 训练: `False` |

In [4]:
# ==================== DataLoader 使用 ====================
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

# 加载数据
transform = transforms.ToTensor()
test_data = torchvision.datasets.CIFAR10(
    root="./dataSet", train=False,
    transform=transform, download=True
)

# 创建 DataLoader
test_loader = DataLoader(
    dataset=test_data,
    batch_size=64,        # 每批 64 张
    shuffle=False,        # 测试时不打乱
    num_workers=0,        # Windows 设为 0
    drop_last=False       # 保留最后批次
)

print(f"数据集大小: {len(test_data)}")
print(f"批次数量:   {len(test_loader)}")
print(f"每批形状:   [batch_size, C=3, H=32, W=32]")

# 遍历 DataLoader
for batch_idx, (imgs, labels) in enumerate(test_loader):
    print(f"批次 {batch_idx}: imgs shape = {imgs.shape}, labels = {labels.tolist()[:5]}...")
    if batch_idx >= 2:  # 只演示前 3 批
        break

print("\n✅ DataLoader 遍历完成")

数据集大小: 10000
批次数量:   157
每批形状:   [batch_size, C=3, H=32, W=32]
批次 0: imgs shape = torch.Size([64, 3, 32, 32]), labels = [3, 8, 8, 0, 6]...
批次 1: imgs shape = torch.Size([64, 3, 32, 32]), labels = [6, 2, 1, 2, 3]...
批次 2: imgs shape = torch.Size([64, 3, 32, 32]), labels = [5, 2, 4, 1, 8]...

✅ DataLoader 遍历完成


### 4.2 nn.Module 的 `__call__` 机制

> **为什么 `model(x)` 而不是 `model.forward(x)` ？**

因为 `nn.Module` 内部实现了 `__call__` 方法，它在调用 `forward()` 前后自动执行 hooks、参数管理、设备切换等操作。

In [5]:
# __call__ 演示
class Person:
    def __call__(self, name):
        print(f"__call__: Hello, {name}!")  # 实例可像函数一样调用
    
    def hello(self, name):
        print(f"hello:   Hello, {name}!")

p = Person()
p.hello("张三")   # 普通方法调用
p("张三")         # __call__ 调用（实例像函数）

hello:   Hello, 张三!
__call__: Hello, 张三!


---

## 5. 总结与速查

### 数据处理标准流水线

```python
# 1. 定义变换
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5])
])

# 2. 创建数据集
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

# 3. 创建 DataLoader
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

# 4. 遍历使用
for imgs, labels in dataloader:
    # imgs: [64, 3, 224, 224]
    pass
```

**下一步**: 学习 [神经网络与完整训练](./03-神经网络与完整训练.ipynb) →